In [2]:
from func_steinhaurer import *
from _functions4plasma import *

import math
import numpy as np 
import sympy as sp 
from scipy.optimize import root
import matplotlib.pyplot as plt 

Lconv = 1e3
acc = 5
# Parameters / Domain Definitions 
Rw          = 6.1/Lconv                 # Wall Radius [m]
Rc          = Rw
zLen        = 200/Lconv                 # Liner length [m]

B0  = 30
sig = f = 1.5                           # Flare parameter; adjustable parameter

T           = np.array([50])            # Temp in [eV]
TK          = T * eV/kb                 # Temp number in [K]

Rmax        = Rw                        # maximum r-domain value [m]
Rmin        = 0                         # minimum r-domain value [m]
Nr          = 1000                      # number of points in r-direction [\]
dr          = (Rmax - Rmin) / Nr        # radial differential [m]
Zmax        = zLen / 2                  # maximum z-domain value [m]
Zmin        = -zLen / 2                 # minimum z-domain value [m]
Nz          = 1000                      # number of points in z-direction [\]
dz          = (Zmax - Zmin) / Nz        # axial differential [m]
domx        = 1.4                       # domain multiplier to make plotting modifications easier
domy        = 0.8                       # domain multiplier to make plotting modifications easier
Rd          = Rmax * domx               # domain radius [m]
Zd          = Zmax * domy               # domain z-length [m]
r = np.arange(-Rd, Rd+dr, dr)
z = np.arange(-Zd, Zd+dz, dz)    
r_mesh, z_mesh = np.meshgrid(r, z, indexing='ij')


conI = True             # Varying Xs and Varying Elongation
conII = False           # Constant Xs and Varying Elongation
conIII = False          # Varying Xs and Constant Elongation

if conI:
    elong_arr   = np.linspace(1,10,num = acc)
    Xs_arr      = np.linspace(0.3, 0.9,num = acc)
    a_arr       = Rc*Xs_arr
    b_arr       =  np.multiply(a_arr, elong_arr)
    Bw_arr      = B0 / (1-(Xs_arr)**2)
elif conII:
    elong_arr   = np.linspace(1,10,num = acc)
    Xs          = 0.6
    a           = Rc*Xs
    b_arr       =  a*elong_arr
    Bw          = B0 / (1 - Xs**2)
elif conIII: 
    E           = 4.5
    Xs_arr      = np.linspace(0.3, 0.9,num = acc)
    a_arr       = Rc*Xs_arr
    b_arr       = a_arr*E
    Bw_arr = B0 / (1-(Xs_arr)**2)
else:
    sys.exit()


def get_flux_contours(psi, R, Z, psi_level_mag, return_all=False):
    # returns rz 
    # 1) Build a tiny OFF‐SCREEN figure, extract contours, then close it:
    plt.ioff()                     # turn off interactive showing
    fig, ax = plt.subplots(figsize=(0.1,0.1))  
    cs = ax.contour(R, Z, psi, levels=[psi_level_mag])
    plt.close(fig)                 # immediately close it so nothing pops up

    # 2) Now extract the contour segments from cs:
    try:
        idx = list(cs.levels).index(psi_level_mag)
    except ValueError:
        raise RuntimeError(f"No ψ={psi_level_mag} level found in cs.levels={cs.levels}")
    segs = cs.allsegs[idx]
    if not segs:
        raise RuntimeError(f"No ψ={psi_level_mag} contour found in the domain.")

    # 3) Convert each Nx2 array into (r_i, z_i) loops exactly as you had:
    loops = []
    for seg in segs:
        verts = np.asarray(seg)    # shape=(Npts,2)
        r_i = verts[:,0].copy()
        z_i = verts[:,1].copy()
        loops.append((r_i, z_i))

    if return_all:
        return loops

    longest = max(loops, key=lambda pair: pair[0].shape[0])
    return longest


In [4]:
e_params    = []
mask = list(range(0, acc))
         
Br_ext_arr          = []
Br_int_arr          = []
Bz_ext_arr          = []
Bz_int_arr          = []

for i, E in enumerate(elong_arr):
        a   = a_arr[i]
        b   = b_arr[i]
        Xs  = Xs_arr[i]
        Bw  = Bw_arr[i]
        Br_int_arr.append((1/r_mesh) * internal_dpsi__dz_sporer(r_mesh, z_mesh, a, b, Bw, Xs))
        Bz_int_arr.append(-(1/r_mesh) * internal_dpsi__dr_sporer(r_mesh, z_mesh, a, b, Bw, Xs, f))

        initial_guess       = [Bw, Bw/3, Bw/5, 0.9]
        result              = root(external_E_params, initial_guess, args= (1/E, Xs, sig))

        if result.success: 
            E0, E1, E2, E3  = result.x
            e_params.append([E0, E1, E2, E3])
            psi_int = (internal_psi_sporer(r_mesh, z_mesh, a, b, Bw, Xs, f))
            psi_ext = (external_psi(r_mesh, z_mesh, Bw, a, b, E0, E1, E2, E3))
                             # is applied to create a full psi(r,z) [T*m^2]
            rz_int = get_flux_contours(psi_int, r_mesh, z_mesh, 0, return_all=False)
            rz_ext = get_flux_contours(psi_ext, r_mesh, z_mesh, 0, return_all=False)

            fig, ax = plt.subplots()
            ax.plot(rz_int[0], rz_int[1], label = r"$\psi_{int}$")
            ax.plot(rz_ext[0], rz_ext[1], label = r"$\psi_{ext}$", linestyle=":")
            ax.set_aspect('equal')
            ax.legend()
            fig.savefig(f"contour{i}")

            dpsi__dr_ext    = external_dpsi__dr(r_mesh, z_mesh, Bw, a, b, E0, E1, E2, E3)       # [T*m^2]; gradient of
            dpsi__dz_ext    = external_dpsi__dz(r_mesh, z_mesh, Bw, a, b, E0, E1, E2, E3)       # external magnetic flux
        
            Br_ext_arr.append(-(1/r_mesh) * dpsi__dz_ext)            # radial magnetic field [T]          #[T*m]
            Bz_ext_arr.append((1/z_mesh) * dpsi__dr_ext)             # axial magnetic field [T]
        else: 
            mask.remove(i)
